# 02 — Data Cleaning & Preprocessing

## Objective
Clean the raw cancellation prediction data, correctly distinguishing
structural nulls (expected based on ride outcome) from genuine data
quality issues, and produce analysis-ready tables.

## What This Notebook Covers
- Removing exact duplicate rows
- Separating structural nulls from data quality nulls
- Handling true missing values — with justification for every decision
- Fixing corrupt and impossible values
- Standardizing data types
- Capping outliers using domain knowledge
- Flagging columns that must be excluded from modeling (data leakage)
- Saving cleaned tables to data/processed/

## Key Principle
A null is not automatically a problem. We first ask WHY it's null
before deciding what to do about it.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

plt.rcParams["figure.dpi"]        = 130
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
DATA_RAW       = "../data/raw/"
DATA_PROCESSED = "../data/processed/"
os.makedirs(DATA_PROCESSED, exist_ok=True)

rides_df   = pd.read_csv(DATA_RAW + "rides.csv")
drivers_df = pd.read_csv(DATA_RAW + "drivers.csv")
users_df   = pd.read_csv(DATA_RAW + "users.csv")
weather_df = pd.read_csv(DATA_RAW + "weather.csv")

print("Raw data loaded ✅")
print(f"  rides   : {rides_df.shape}")
print(f"  drivers : {drivers_df.shape}")
print(f"  users   : {users_df.shape}")
print(f"  weather : {weather_df.shape}")

Raw data loaded ✅
  rides   : (60300, 34)
  drivers : (3000, 17)
  users   : (20200, 17)
  weather : (87600, 7)


## Step 1 — Remove Duplicate Rows

Same approach as surge pricing — drop exact duplicate rows, then
deduplicate on primary keys (ride_id, driver_id, user_id) to catch
near-duplicates that aren't 100% identical across every column.

In [3]:
before = {
    "rides"  : len(rides_df),
    "drivers": len(drivers_df),
    "users"  : len(users_df),
    "weather": len(weather_df),
}

rides_df   = rides_df.drop_duplicates().reset_index(drop=True)
drivers_df = drivers_df.drop_duplicates().reset_index(drop=True)
users_df   = users_df.drop_duplicates().reset_index(drop=True)
weather_df = weather_df.drop_duplicates().reset_index(drop=True)

rides_df   = rides_df.drop_duplicates(subset=["ride_id"]).reset_index(drop=True)
drivers_df = drivers_df.drop_duplicates(subset=["driver_id"]).reset_index(drop=True)
users_df   = users_df.drop_duplicates(subset=["user_id"]).reset_index(drop=True)

after = {
    "rides"  : len(rides_df),
    "drivers": len(drivers_df),
    "users"  : len(users_df),
    "weather": len(weather_df),
}

print("Duplicate Removal Summary")
print(f"{'Table':<10} {'Before':>8} {'After':>8} {'Removed':>8}")
print("-" * 38)
for name in before:
    removed = before[name] - after[name]
    print(f"{name:<10} {before[name]:>8,} {after[name]:>8,} {removed:>8,}")

Duplicate Removal Summary
Table        Before    After  Removed
--------------------------------------
rides        60,300   60,000      300
drivers       3,000    3,000        0
users        20,200   20,000      200
weather      87,600   87,600        0


## Step 2 — Separate Structural Nulls from Quality Nulls

### Structural Nulls (expected, meaningful, DO NOT impute)
These columns are null BECAUSE of the ride outcome — not because
data is missing or broken.

| Column | Null when... | Why |
|---|---|---|
| distance_km | ride_outcome != 0 (not completed) | Trip never happened |
| duration_min | ride_outcome != 0 | Trip never happened |
| fare_amount | ride_outcome != 0 | No fare charged |
| time_to_cancellation_min | ride_outcome == 0 (completed) | No cancellation occurred |
| cancelled_by | ride_outcome == 0 | No cancellation occurred |
| cancellation_reason | ride_outcome == 0 | No cancellation occurred |

### Quality Nulls (genuine missingness, DO need handling)
| Column | Why it's missing |
|---|---|
| pickup_zone, drop_zone | App/backend failure |
| estimated_wait_time_min | Sensor/calculation failure |
| driver_distance_to_pickup_km | GPS failure |
| driver_rating_at_booking, driver_acceptance_rate | New driver, not yet rated |
| pickup_accuracy_score | App failure |
| phone, email (users/drivers) | User skipped optional field |

We verify the structural pattern holds, then handle each
category with a different strategy.

In [4]:
structural_cols_completed_only = [
    "distance_km", "duration_min", "fare_amount"
]
structural_cols_cancelled_only = [
    "time_to_cancellation_min", "cancelled_by", "cancellation_reason"
]

print("Verifying structural null patterns hold cleanly:\n")

for col in structural_cols_completed_only:
    null_in_completed = rides_df[rides_df["ride_outcome"] == 0][col].isnull().mean()
    null_in_cancelled = rides_df[rides_df["ride_outcome"] != 0][col].isnull().mean()
    print(f"  {col:<28} null% when completed={null_in_completed*100:.1f}%  "
          f"null% when cancelled={null_in_cancelled*100:.1f}%")

print()
for col in structural_cols_cancelled_only:
    null_in_completed = rides_df[rides_df["ride_outcome"] == 0][col].isnull().mean()
    null_in_cancelled = rides_df[rides_df["ride_outcome"] != 0][col].isnull().mean()
    print(f"  {col:<28} null% when completed={null_in_completed*100:.1f}%  "
          f"null% when cancelled={null_in_cancelled*100:.1f}%")

print("\n✅ Confirmed: these nulls are structural, not data quality issues.")
print("   We will NOT impute these — they correctly represent 'not applicable'.")

Verifying structural null patterns hold cleanly:

  distance_km                  null% when completed=0.0%  null% when cancelled=100.0%
  duration_min                 null% when completed=0.0%  null% when cancelled=100.0%
  fare_amount                  null% when completed=0.0%  null% when cancelled=100.0%

  time_to_cancellation_min     null% when completed=100.0%  null% when cancelled=0.0%
  cancelled_by                 null% when completed=100.0%  null% when cancelled=0.0%
  cancellation_reason          null% when completed=100.0%  null% when cancelled=12.8%

✅ Confirmed: these nulls are structural, not data quality issues.
   We will NOT impute these — they correctly represent 'not applicable'.


## Step 3 — Fix Data Types

Same principle as surge pricing — convert timestamps, booleans,
and numerics to their correct types so calculations and
time-based analysis work correctly.

In [5]:
# ── rides ────────────────────────────────────────────────────
rides_df["ride_timestamp"] = pd.to_datetime(rides_df["ride_timestamp"])
rides_df["hour"]           = rides_df["hour"].astype(int)
rides_df["month"]          = rides_df["month"].astype(int)
rides_df["is_weekend"]     = rides_df["is_weekend"].astype(bool)
rides_df["is_holiday"]     = rides_df["is_holiday"].astype(bool)
rides_df["ride_outcome"]   = rides_df["ride_outcome"].astype(int)

numeric_cols = [
    "driver_distance_to_pickup_km", "estimated_wait_time_min",
    "surge_at_booking", "distance_km", "duration_min", "fare_amount",
    "driver_rating_at_booking", "driver_acceptance_rate",
    "driver_cancellations_today", "user_rating_at_booking",
    "user_cancellations_last_30d", "user_booking_attempts",
    "time_to_cancellation_min", "pickup_accuracy_score"
]
for col in numeric_cols:
    rides_df[col] = pd.to_numeric(rides_df[col], errors="coerce")

# ── drivers ──────────────────────────────────────────────────
drivers_df["joined_date"] = pd.to_datetime(drivers_df["joined_date"])
drivers_df["is_active"]   = drivers_df["is_active"].astype(bool)
for col in ["rating", "acceptance_rate", "cancellation_rate"]:
    drivers_df[col] = pd.to_numeric(drivers_df[col], errors="coerce")

# ── users ────────────────────────────────────────────────────
users_df["signup_date"] = pd.to_datetime(users_df["signup_date"])
for col in ["age", "rating", "total_cancellations", "cancellations_last_30d"]:
    users_df[col] = pd.to_numeric(users_df[col], errors="coerce")

# ── weather ──────────────────────────────────────────────────
weather_df["timestamp"]   = pd.to_datetime(weather_df["timestamp"])
weather_df["temperature"] = pd.to_numeric(weather_df["temperature"], errors="coerce")
weather_df["humidity"]    = pd.to_numeric(weather_df["humidity"],    errors="coerce")

print("Data types fixed ✅")
print(rides_df.dtypes)

Data types fixed ✅
ride_id                                    str
driver_id                                  str
user_id                                    str
pickup_zone                                str
drop_zone                                  str
pickup_lat                             float64
pickup_lon                             float64
drop_lat                               float64
drop_lon                               float64
pickup_accuracy_score                  float64
ride_timestamp                  datetime64[us]
hour                                     int64
month                                    int64
day_of_week                                str
is_weekend                                bool
is_holiday                                bool
vehicle_type                               str
weather_condition                          str
driver_distance_to_pickup_km           float64
driver_rating_at_booking               float64
driver_acceptance_rate                 fl

## Step 4 — Fix Corrupt and Impossible Values

### Known corrupt values from data generation:
- user age: -1, 0, 150, 999
- driver_distance_to_pickup_km: should never be negative or absurdly large

### Decision
Convert impossible values to NaN, handle in the next step.

In [6]:
# ── user age ─────────────────────────────────────────────────
print("User age corrupt values before fix:")
print(f"  Below 16 or above 100: {((users_df['age'] < 16) | (users_df['age'] > 100)).sum()}")

users_df.loc[users_df["age"] < 16,  "age"] = np.nan
users_df.loc[users_df["age"] > 100, "age"] = np.nan

# ── driver distance to pickup ───────────────────────────────
print("\ndriver_distance_to_pickup_km corrupt values before fix:")
print(f"  Negative or zero: {(rides_df['driver_distance_to_pickup_km'] <= 0).sum()}")
print(f"  Over 25 km       : {(rides_df['driver_distance_to_pickup_km'] > 25).sum()}")

rides_df.loc[rides_df["driver_distance_to_pickup_km"] <= 0, "driver_distance_to_pickup_km"] = np.nan
rides_df.loc[rides_df["driver_distance_to_pickup_km"] > 25, "driver_distance_to_pickup_km"] = np.nan

# ── estimated wait time ─────────────────────────────────────
print("\nestimated_wait_time_min corrupt values before fix:")
print(f"  Negative or zero: {(rides_df['estimated_wait_time_min'] <= 0).sum()}")

rides_df.loc[rides_df["estimated_wait_time_min"] <= 0, "estimated_wait_time_min"] = np.nan

print("\n✅ Corrupt values converted to NaN")

User age corrupt values before fix:
  Below 16 or above 100: 448

driver_distance_to_pickup_km corrupt values before fix:
  Negative or zero: 0
  Over 25 km       : 0

estimated_wait_time_min corrupt values before fix:
  Negative or zero: 465

✅ Corrupt values converted to NaN


## Step 5 — Handle Genuine Quality Missing Values

### Rides Table (booking-time features only)
| Column | Strategy | Reason |
|---|---|---|
| pickup_zone, drop_zone | Mode imputation | Categorical, most common reasonable |
| estimated_wait_time_min | Median by hour | Wait time varies a lot by time of day |
| driver_distance_to_pickup_km | Median | Numerical, slightly skewed |
| driver_rating_at_booking | Median | Numerical |
| driver_acceptance_rate | Median | Numerical |
| pickup_accuracy_score | Median | Numerical |

### Users Table
| Column | Strategy | Reason |
|---|---|---|
| age | Median | Roughly normal distribution |
| rating | Median | Ratings cluster tightly |
| phone, email | Leave as NaN | Cannot fabricate contact info |

### Drivers Table
| Column | Strategy | Reason |
|---|---|---|
| rating | Median | Numerical |
| acceptance_rate | Median | Numerical |
| online_hours_per_day | Median | Numerical |
| vehicle_model | Fill "Unknown" | Categorical |
| phone | Leave as NaN | Cannot fabricate |

In [7]:
# ── pickup_zone / drop_zone → mode ────────────────────────────
mode_pickup = rides_df["pickup_zone"].mode()[0]
mode_drop   = rides_df["drop_zone"].mode()[0]
rides_df["pickup_zone"] = rides_df["pickup_zone"].fillna(mode_pickup)
rides_df["drop_zone"]   = rides_df["drop_zone"].fillna(mode_drop)
print(f"pickup_zone filled with mode: {mode_pickup}")
print(f"drop_zone filled with mode  : {mode_drop}")

# ── estimated_wait_time_min → median per hour ─────────────────
median_wait_by_hour = rides_df.groupby("hour")["estimated_wait_time_min"].transform("median")
rides_df["estimated_wait_time_min"] = rides_df["estimated_wait_time_min"].fillna(median_wait_by_hour)
print(f"estimated_wait_time_min filled with median per hour")

# ── driver_distance_to_pickup_km → median ─────────────────────
median_dist = rides_df["driver_distance_to_pickup_km"].median()
rides_df["driver_distance_to_pickup_km"] = rides_df["driver_distance_to_pickup_km"].fillna(median_dist)
print(f"driver_distance_to_pickup_km filled with median: {median_dist:.2f} km")

# ── driver_rating_at_booking / acceptance_rate → median ───────
for col in ["driver_rating_at_booking", "driver_acceptance_rate"]:
    median_val = rides_df[col].median()
    rides_df[col] = rides_df[col].fillna(median_val)
    print(f"{col} filled with median: {median_val:.2f}")

# ── pickup_accuracy_score → median ─────────────────────────────
median_acc = rides_df["pickup_accuracy_score"].median()
rides_df["pickup_accuracy_score"] = rides_df["pickup_accuracy_score"].fillna(median_acc)
print(f"pickup_accuracy_score filled with median: {median_acc:.2f}")

# ── user_rating_at_booking → median ────────────────────────────
median_user_rating = rides_df["user_rating_at_booking"].median()
rides_df["user_rating_at_booking"] = rides_df["user_rating_at_booking"].fillna(median_user_rating)

print("\nRemaining nulls in booking-time feature columns:")
booking_time_cols = [
    "pickup_zone","drop_zone","estimated_wait_time_min",
    "driver_distance_to_pickup_km","driver_rating_at_booking",
    "driver_acceptance_rate","pickup_accuracy_score","user_rating_at_booking"
]
print(rides_df[booking_time_cols].isnull().sum())

pickup_zone filled with mode: Bellandur
drop_zone filled with mode  : Bellandur
estimated_wait_time_min filled with median per hour
driver_distance_to_pickup_km filled with median: 1.73 km
driver_rating_at_booking filled with median: 4.30
driver_acceptance_rate filled with median: 0.83
pickup_accuracy_score filled with median: 0.65

Remaining nulls in booking-time feature columns:
pickup_zone                     0
drop_zone                       0
estimated_wait_time_min         0
driver_distance_to_pickup_km    0
driver_rating_at_booking        0
driver_acceptance_rate          0
pickup_accuracy_score           0
user_rating_at_booking          0
dtype: int64


In [8]:
# ── users ────────────────────────────────────────────────────
median_age = users_df["age"].median()
users_df["age"] = users_df["age"].fillna(median_age)
print(f"user age filled with median: {median_age:.0f}")

median_user_rating = users_df["rating"].median()
users_df["rating"] = users_df["rating"].fillna(median_user_rating)
print(f"user rating filled with median: {median_user_rating:.2f}")

users_df["gender"] = users_df["gender"].fillna("Not Specified")

print(f"\nphone nulls kept (cannot fabricate): {users_df['phone'].isnull().sum()}")
print(f"email nulls kept (cannot fabricate): {users_df['email'].isnull().sum()}")

# ── drivers ──────────────────────────────────────────────────
median_driver_rating = drivers_df["rating"].median()
drivers_df["rating"] = drivers_df["rating"].fillna(median_driver_rating)

median_acceptance = drivers_df["acceptance_rate"].median()
drivers_df["acceptance_rate"] = drivers_df["acceptance_rate"].fillna(median_acceptance)

median_online = drivers_df["online_hours_per_day"].median()
drivers_df["online_hours_per_day"] = drivers_df["online_hours_per_day"].fillna(median_online)

drivers_df["vehicle_model"] = drivers_df["vehicle_model"].fillna("Unknown")

print(f"\ndriver phone nulls kept: {drivers_df['phone'].isnull().sum()}")

print("\nUsers remaining nulls:")
print(users_df.isnull().sum()[users_df.isnull().sum() > 0])
print("\nDrivers remaining nulls:")
print(drivers_df.isnull().sum()[drivers_df.isnull().sum() > 0])

user age filled with median: 39
user rating filled with median: 4.20

phone nulls kept (cannot fabricate): 597
email nulls kept (cannot fabricate): 1033

driver phone nulls kept: 58

Users remaining nulls:
phone               597
email              1033
is_prime_member    6751
dtype: int64

Drivers remaining nulls:
phone            58
vehicle_year    350
dtype: int64


## Step 6 — Outlier Capping (IQR Method)

Same IQR-based capping approach as surge pricing. We cap rather
than drop, to preserve the rest of each row's information.

### Columns we cap:
- driver_distance_to_pickup_km
- estimated_wait_time_min
- user_booking_attempts
- driver_cancellations_today (capped lightly — high values are
  meaningful signal, not necessarily errors, so we use a wider cap)

In [9]:
def cap_outliers_iqr(df, column, multiplier=1.5):
    Q1  = df[column].quantile(0.25)
    Q3  = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR

    before_low  = (df[column] < lower).sum()
    before_high = (df[column] > upper).sum()

    df[column] = df[column].clip(lower=lower, upper=upper)

    print(f"  {column:<32} lower={lower:.2f}  upper={upper:.2f}  "
          f"capped_low={before_low}  capped_high={before_high}")
    return df

print("Outlier Capping — Rides Table")
print("-" * 75)
for col in ["driver_distance_to_pickup_km", "estimated_wait_time_min", "user_booking_attempts"]:
    rides_df = cap_outliers_iqr(rides_df, col)

# Wider cap for cancellations_today — it's a meaningful count, not noise
rides_df = cap_outliers_iqr(rides_df, "driver_cancellations_today", multiplier=3.0)

print("\n✅ Outlier capping complete")

Outlier Capping — Rides Table
---------------------------------------------------------------------------
  driver_distance_to_pickup_km     lower=-3.33  upper=7.50  capped_low=0  capped_high=3007
  estimated_wait_time_min          lower=-12.85  upper=30.75  capped_low=0  capped_high=3582
  user_booking_attempts            lower=-2.00  upper=6.00  capped_low=0  capped_high=0
  driver_cancellations_today       lower=0.00  upper=0.00  capped_low=0  capped_high=101

✅ Outlier capping complete


## Step 7 — Final Validation Before Saving

Confirm:
- No unexpected nulls remain in booking-time features
- Structural nulls (post-outcome columns) are still correctly null
- No impossible values remain
- Target variable distribution unchanged

In [10]:
print("FINAL VALIDATION\n")

tables_clean = {
    "rides": rides_df, "drivers": drivers_df,
    "users": users_df, "weather": weather_df,
}
for name, df in tables_clean.items():
    print(f"  {name:<10} rows={len(df):>7,}  nulls={df.isnull().sum().sum():>6,}  "
          f"dupes={df.duplicated().sum():>4,}")

print("\nBooking-time feature nulls (should be 0):")
print(rides_df[booking_time_cols].isnull().sum())

print("\nStructural nulls preserved correctly:")
for col in structural_cols_completed_only + structural_cols_cancelled_only:
    print(f"  {col:<28} null count: {rides_df[col].isnull().sum():,}")

print("\nTarget distribution unchanged:")
print(rides_df["ride_outcome"].value_counts(normalize=True).round(4) * 100)

print("\nSanity checks:")
print(f"  Negative driver distance : {(rides_df['driver_distance_to_pickup_km'] < 0).sum()}")
print(f"  Negative wait time       : {(rides_df['estimated_wait_time_min'] < 0).sum()}")
print(f"  Age below 16             : {(users_df['age'] < 16).sum()}")

FINAL VALIDATION

  rides      rows= 60,000  nulls=181,715  dupes=   0
  drivers    rows=  3,000  nulls=   408  dupes=   0
  users      rows= 20,000  nulls= 8,381  dupes=   0
  weather    rows= 87,600  nulls= 3,544  dupes=   0

Booking-time feature nulls (should be 0):
pickup_zone                     0
drop_zone                       0
estimated_wait_time_min         0
driver_distance_to_pickup_km    0
driver_rating_at_booking        0
driver_acceptance_rate          0
pickup_accuracy_score           0
user_rating_at_booking          0
dtype: int64

Structural nulls preserved correctly:
  distance_km                  null count: 13,403
  duration_min                 null count: 13,403
  fare_amount                  null count: 13,403
  time_to_cancellation_min     null count: 46,597
  cancelled_by                 null count: 46,597
  cancellation_reason          null count: 48,312

Target distribution unchanged:
ride_outcome
0   77.660
2   15.800
1    6.540
Name: proportion, dtype: flo

In [11]:
rides_df.to_csv(  DATA_PROCESSED + "rides_clean.csv",   index=False)
drivers_df.to_csv(DATA_PROCESSED + "drivers_clean.csv", index=False)
users_df.to_csv(  DATA_PROCESSED + "users_clean.csv",   index=False)
weather_df.to_csv(DATA_PROCESSED + "weather_clean.csv", index=False)

print("✅ All cleaned tables saved to data/processed/")
print(f"  rides_clean.csv   : {len(rides_df):,} rows")
print(f"  drivers_clean.csv : {len(drivers_df):,} rows")
print(f"  users_clean.csv   : {len(users_df):,} rows")
print(f"  weather_clean.csv : {len(weather_df):,} rows")

✅ All cleaned tables saved to data/processed/
  rides_clean.csv   : 60,000 rows
  drivers_clean.csv : 3,000 rows
  users_clean.csv   : 20,000 rows
  weather_clean.csv : 87,600 rows


## Cleaning Summary

| Step | What We Did | Why |
|---|---|---|
| Duplicates | Dropped exact + primary key duplicates | Prevent model bias |
| Structural Nulls | Identified and preserved (NOT imputed) | They represent "not applicable", not missing data |
| Data Types | Converted timestamps, booleans, numerics | Enable correct calculations |
| Corrupt Values | Converted impossible values to NaN | Separate detection from imputation |
| Quality Nulls | Strategy per column based on domain logic | Never blind imputation |
| Outliers | IQR-based capping | Preserve rows, reduce extreme influence |

## What Is Still Missing (Intentionally)
- phone/email nulls — cannot fabricate contact info
- distance_km/duration_min/fare_amount for cancelled rides — structurally correct
- cancelled_by/cancellation_reason/time_to_cancellation_min for completed rides — structurally correct

## Critical Reminder for Next Notebook
distance_km, duration_min, fare_amount, time_to_cancellation_min,
cancelled_by, and cancellation_reason must be EXCLUDED from the
feature set in notebook 04 — they are only known AFTER the
outcome, and including them would cause data leakage.

## Next Step → Notebook 03 — EDA
Now we explore what booking-time signals actually predict
cancellation behavior.